# RAGify Benchmark - Google Colab Setup

This notebook sets up the complete environment for running the RAGify benchmark in Google Colab.

## Prerequisites
- Google Colab with GPU recommended (for better Ollama performance)
- Google Drive for persistent storage (optional)
- ~10GB free space recommended

## Step 1: Install Ollama

In [ ]:
# Install Ollama
!curl -fsSL https://ollama.com/install.sh | sh

# Add to PATH
import os
os.environ['PATH'] = f"{os.environ['PATH']}:/root/.local/bin"

# Verify installation
!ollama --version

## Step 2: Start Ollama Service

In [ ]:
# Start Ollama in background
!ollama serve &

# Wait for service to start
import time
time.sleep(5)

# Check if running
!pgrep -f ollama

## Step 3: Download Ollama Models

In [ ]:
# Pull the LLM model (e.g., llama3.2 or your preferred model)
!ollama pull llama3.2

# Pull embedding model (nomic-embed-text is recommended for embeddings)
!ollama pull nomic-embed-text

# List installed models
!ollama list

## Step 4: Install Docker and Docker Compose

In [ ]:
# Install Docker
!apt-get update
!apt-get install -y docker.io docker-compose

# Start Docker daemon
!dockerd &
time.sleep(5)

# Verify Docker
!docker --version

## Step 5: Install uv (Python Package Manager)

In [ ]:
# Install uv
!curl -LsSf https://astral.sh/uv/install.sh | sh

# Add to PATH
os.environ['PATH'] = f"{os.environ['PATH']}:/root/.local/bin"

# Verify uv
!uv --version

## Step 6: Clone/Setup Project and Configure Environment

In [ ]:
# Mount Google Drive (optional - for persistent storage)
from google.colab import drive
drive.mount('/content/drive')

# Or clone the repository
!git clone https://github.com/your-repo/ragify.git /content/ragify
cd /content/ragify

## Step 7: Create Environment File

In [ ]:
# Create .env file with default values
env_content = """# MongoDB
MONGO_PORT=27017
MONGO_INITDB_ROOT_USERNAME=myuser
MONGO_INITDB_ROOT_PASSWORD=mypassword

# Qdrant
QDRANT_HTTP_PORT=6333
QDRANT_GRPC_PORT=6334

# Grobid
GROBID_PORT=8070

# Ollama
OLLAMA_BASE_URL=http://localhost:11434

# API
PORT=1337
"""

with open('/content/ragify/infra/.env', 'w') as f:
    f.write(env_content)

print("Environment file created!")

## Step 8: Update docker-compose.yml with Grobid

In [ ]:
# Read current docker-compose.yml
with open('/content/ragify/infra/docker/docker-compose.yml', 'r') as f:
    content = f.read()

# Check if grobid already exists, if not add it
if 'grobid' not in content.lower():
    # Add grobid service before volumes section
    grobid_service = '''  grobid:
    container_name: ragify-grobid
    image: grobid/grobid:0.9.0-crf
    ports:
      - "${GROBID_PORT:-8070}:8070"
    ulimits:
      core:
        soft: 0
        hard: 0
    init: true
    restart: unless-stopped

'''
    content = content.replace('volumes:', grobid_service + 'volumes:')
    
    with open('/content/ragify/infra/docker/docker-compose.yml', 'w') as f:
        f.write(content)
    print("Grobid service added to docker-compose.yml")
else:
    print("Grobid service already exists in docker-compose.yml")

## Step 9: Run Docker Compose

In [ ]:
# Navigate to infra/docker directory
import os
os.chdir('/content/ragify/infra/docker')

# Run docker compose using the run.sh script
# First, let's modify run.sh to work in Colab environment

# Create a modified start script
start_script = '''#!/bin/bash
cd "$(dirname "$0")"
docker-compose --env-file=../../infra/.env up -d
'''

with open('/content/ragify/start_colab.sh', 'w') as f:
    f.write(start_script)

os.chmod('/content/ragify/start_colab.sh', 0o755)

# Run docker compose
!docker-compose --env-file=/content/ragify/infra/.env up -d

# Check running containers
!docker ps

## Step 10: Verify Services

In [ ]:
# Check if all services are running
!docker ps

# Check logs for each service
!docker logs ragify-db --tail 10
!docker logs ragify_qdrant --tail 10
!docker logs ragify-grobid --tail 10

## Step 11: Install Python Dependencies

In [ ]:
# Install Python dependencies using uv
!cd /content/ragify && uv pip install -r requirements.txt

# Or install specific packages needed for benchmark
!uv pip install pymongo qdrant-client ollama pypdf requests pandas numpy

## Step 12: Run the Benchmark

In [ ]:
# Navigate to project directory
import os
os.chdir('/content/ragify')

# Check for benchmark script
import glob
benchmark_files = glob.glob('**/benchmark*.py', recursive=True)
print("Benchmark files found:", benchmark_files)

# Run benchmark if exists
if benchmark_files:
    !python {benchmark_files[0]}
else:
    print("No benchmark script found. Please add your benchmark script.")

## Utility: Stop All Services

In [ ]:
# Stop all Docker containers
!docker-compose --env-file=/content/ragify/infra/.env down

# Stop Ollama
!pkill -f ollama

print("All services stopped!")

## Utility: Check Service Health

In [ ]:
import requests

def check_service(url, name):
    try:
        r = requests.get(url, timeout=5)
        print(f"{name}: ✓ (Status: {r.status_code})")
    except Exception as e:
        print(f"{name}: ✗ ({str(e))})")

# Check all services
check_service("http://localhost:27017", "MongoDB")
check_service("http://localhost:6333", "Qdrant (HTTP)")
check_service("http://localhost:8070", "Grobid")
check_service("http://localhost:11434", "Ollama")

---

## Notes

- **MongoDB**: Runs on port 27017
- **Qdrant**: HTTP on port 6333, gRPC on port 6334
- **Grobid**: Runs on port 8070
- **Ollama**: Runs on port 11434

All services should be accessible from within the Colab environment.